# 02 · Data Preparation — order-level dataset with leakage-safe features

**Stage 3.** This notebook builds one row per order from the tables selected in `01_data_audit` (orders, items, customers, sellers, products, the compact zip map and payments). Reviews and the translation table are not loaded.

Contents:
* Labels and the time each outcome becomes known (`known_at`).
* Point-in-time seller, route, destination and market history.
* A validated join log.
* The population funnel.
* A feature-by-feature leakage audit.
* A demonstration of what a leaky seller feature would do.
* The provisional chronological split, with purge counts.

All logic lives in `src/olist/` (`dataset.py`, `history.py`, `features.py`, `split.py`). Temporal leakage is checked by `tests/test_history.py` against a brute-force recomputation and a future-perturbation test.

In [1]:
import sys, time, logging
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, psutil
from sklearn.metrics import roc_auc_score, average_precision_score
from olist import config as C
from olist.dataset import build_order_dataset
from olist.features import FEATURES, leakage_audit
from olist.split import assign_split, split_summary, label_known_by, matured_by

logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 120)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
proc = psutil.Process(); T0 = time.perf_counter()
rss = lambda: f"{proc.memory_info().rss/1e6:,.0f} MB"

## 1. Build the dataset (every join is cardinality-checked)

In [2]:
t = time.perf_counter()
df, joins = build_order_dataset()
print(f"built {df.shape} in {time.perf_counter()-t:.1f}s | frame {df.memory_usage(deep=True).sum()/1e6:.0f} MB | RSS {rss()}")
pd.DataFrame(joins).set_index("name")

[orders-customers] 1:1 left on customer_id->customer_id: rows 99441 -> 99441, unmatched left rows 0


[items-products] m:1 left on product_id->product_id: rows 112650 -> 112650, unmatched left rows 0


[items-sellers] m:1 left on seller_id->seller_id: rows 112650 -> 112650, unmatched left rows 0


[items-orders] m:1 left on order_id->order_id: rows 112650 -> 112650, unmatched left rows 0


[orders-items_agg] 1:1 left on order_id->order_id: rows 99441 -> 99441, unmatched left rows 775


[orders-payments_agg] 1:1 left on order_id->order_id: rows 99441 -> 99441, unmatched left rows 1


[orders-history] 1:1 left on order_id->order_id: rows 99441 -> 99441, unmatched left rows 775


order dataset (98666, 79) built in 18.0s


built (98666, 79) in 18.0s | frame 100 MB | RSS 351 MB


,rows_before,rows_after,unmatched_left,validate
name,,,,
orders-customers,99441,99441,0,1:1
orders-items_agg,99441,99441,775,1:1
orders-payments_agg,99441,99441,1,1:1
orders-history,99441,99441,775,1:1


Each order-level join keeps exactly 99,441 rows. The unmatched rows are the 775 orders without items (none of them delivered) and one order without a payment record. Orders without items are dropped at the end because they have no basket, so the dataset holds one row per order that has items. The item-level joins (products, sellers, customer coordinates) are logged above and also keep the item row count of 112,650.

## 2. Labels and outcome-known time
| column | definition |
|---|---|
| `late_day` (**primary**) | delivered **and** delivery day > promised day |
| `late_ts` (sensitivity) | delivered **and** delivery timestamp > promised date at 00:00, so delivery *on* the promised day counts as late |
| `promise_failed` (sensitivity) | `late_day` for delivered orders, plus 1 for never-delivered, non-cancelled orders past their promise |
| `known_at` | `min(delivered_ts, promised_date + 1 day)`, the moment the marketplace learns the outcome. Used only to build history and to purge splits |

In [3]:
pd.crosstab(df.order_status, df.outcome_class, margins=True)

outcome_class,delivered,open_failed,other,All
order_status,,,,
approved,0,2,0,2
canceled,0,0,461,461
delivered,96470,0,8,96478
invoiced,0,312,0,312
processing,0,301,0,301
shipped,0,1106,0,1106
unavailable,0,0,6,6
All,96470,1721,475,98666


## 3. Population funnel

In [4]:
P = "order_purchase_timestamp"
steps = [
    ("orders with items", pd.Series(True, index=df.index)),
    ("outcome observed (delivered)", df.outcome_class.eq("delivered")),
    ("purchase >= 2017-01-01", df.outcome_class.eq("delivered") & (df[P] >= C.POPULATION_START)),
    ("purchase < 2018-08-01", df.outcome_class.eq("delivered") & (df[P] >= C.POPULATION_START) & (df[P] < C.POPULATION_END)),
    ("promised date <= T_OBS  (= eligible_primary)", df.eligible_primary),
]
funnel = pd.DataFrame([{"step": s, "orders": int(m.sum()), "late_day": int(df.loc[m, "late_day"].sum()),
                        "late_%": round(df.loc[m, "late_day"].mean() * 100, 2)} for s, m in steps])
funnel["removed"] = -funnel.orders.diff()
display(funnel)
sens = df[df.eligible_sensitivity]
print(f"sensitivity population (adds open failed orders): {len(sens):,} orders, promise_failed rate {sens.promise_failed.mean():.2%} "
      f"(+{int((sens.outcome_class=='open_failed').sum()):,} positives)")
pop = df[df.eligible_primary].copy()

,step,orders,late_day,late_%,removed
0,orders with items,98666,6534,6.77,NaN
1,outcome observed (delivered),96470,6534,6.77,2196.0
2,purchase >= 2017-01-01,96203,6531,6.79,267.0
3,purchase < 2018-08-01,89852,6138,6.83,6351.0
4,promised date <= T_OBS (= eligible_primary),89795,6138,6.84,57.0


sensitivity population (adds open failed orders): 91,419 orders, promise_failed rate 8.49% (+1,624 positives)


## 4. Data-quality flags inside the population: flagged, not dropped
These problems affect only post-checkout fields, which are never Task A inputs. Dropping the orders would condition the sample on how the order was later processed.

In [5]:
dq = pop.filter(like="dq_").sum().rename("orders").to_frame()
dq["late_rate_%"] = [round(pop.loc[pop[c], "late_day"].mean() * 100, 1) for c in dq.index]
dq

,orders,late_rate_%
dq_carrier_before_purchase,115,1.7
dq_carrier_before_approval,1261,1.7
dq_delivered_before_carrier,19,0.0


## 5. Missing values in model features (primary population)

In [6]:
na = pop[FEATURES.index].isna().sum()
na = na[na > 0].rename("n_missing").to_frame()
na["pct"] = (na.n_missing / len(pop) * 100).round(2)
na["planned treatment (fitted on training data only)"] = [
    "median imputation + missing indicator (category-level median tested in Stage 4)" if "weight" in i or "volume" in i
    else "median imputation; missingness == any_category_missing (same 610 products)" for i in na.index]
na

,n_missing,pct,planned treatment (fitted on training data only)
total_weight_g,16,0.02,median imputation + missing indicator (categor...
max_weight_g,16,0.02,median imputation + missing indicator (categor...
total_volume_cm3,16,0.02,median imputation + missing indicator (categor...
max_volume_cm3,16,0.02,median imputation + missing indicator (categor...
mean_photos,1303,1.45,median imputation; missingness == any_category...
mean_description_length,1303,1.45,median imputation; missingness == any_category...
mean_name_length,1303,1.45,median imputation; missingness == any_category...


In [7]:
print("text stats missing => category missing for every order:", bool((~pop.mean_photos.isna() | pop.any_category_missing.astype(bool)).all()))
print("categorical cardinalities:", {c: pop[c].nunique() for c in FEATURES.index[FEATURES.kind == "cat"]})

text stats missing => category missing for every order: True
categorical cardinalities: {'promise_dow': 5, 'purchase_dow': 7, 'purchase_month': 12, 'main_category': 74, 'customer_state': 27, 'main_seller_state': 22, 'main_payment_type': 4}


## 6. Feature leakage audit
Every candidate column, when it becomes available, and whether it may be a Task A input. Inputs are restricted to `FEATURES`. A test (`tests/test_dataset.py`) asserts that `FEATURES` and `FORBIDDEN` are disjoint.

In [8]:
audit = leakage_audit()
print(audit.task_a_input.value_counts().to_dict(), "| feature groups:", FEATURES.group.value_counts().to_dict())
audit

{'yes': 57, 'no': 21} | feature groups: {'history': 14, 'geography': 10, 'basket': 9, 'product': 9, 'calendar': 7, 'promise': 4, 'payment': 4}


,group,source,known_when,task_a_input
column,,,,
promised_days,promise,orders.estimated - purchase,checkout (estimate shown to customer),yes
promised_business_days,promise,orders + holiday calendar,checkout,yes
holidays_in_promise,promise,orders + holiday calendar,checkout,yes
promise_dow,promise,orders.estimated,checkout,yes
purchase_hour,calendar,orders.purchase,checkout,yes
purchase_dow,calendar,orders.purchase,checkout,yes
purchase_month,calendar,orders.purchase,checkout,yes
purchase_dom,calendar,orders.purchase,checkout,yes
purchase_is_weekend,calendar,orders.purchase,checkout,yes


## 7. History features: cold start over time

In [9]:
pop["ym"] = pop[P].dt.to_period("M")
cs = pop.groupby("ym").agg(orders=("order_id", "size"), seller_cold_start=("seller_cold_start", "mean"),
                           median_seller_resolved=("seller_resolved_n_min", "median"),
                           route_cold=("route_resolved_n_min", lambda s: (s < C.COLD_START_N).mean()),
                           market_late_rate_30d=("market_late_rate_30d", "mean"), actual_late_rate=("late_day", "mean"))
cs.round(3).T

ym,2017-01,2017-02,2017-03,2017-04,2017-05,2017-06,2017-07,2017-08,2017-09,2017-10,2017-11,2017-12,2018-01,2018-02,2018-03,2018-04,2018-05,2018-06,2018-07
orders,750.000,1653.000,2546.000,2303.000,3545.000,3135.000,3872.000,4193.000,4150.000,4478.000,7288.000,5513.000,7069.000,6555.000,7003.000,6798.000,6749.000,6096.000,6099.000
seller_cold_start,0.809,0.581,0.351,0.235,0.201,0.115,0.136,0.138,0.140,0.137,0.130,0.102,0.121,0.099,0.087,0.125,0.119,0.130,0.141
median_seller_resolved,1.000,3.000,9.500,19.000,24.000,35.000,40.000,45.000,51.000,59.000,62.000,68.000,79.000,67.000,85.000,79.000,72.000,75.000,73.000
route_cold,0.300,0.157,0.086,0.048,0.029,0.018,0.018,0.018,0.018,0.017,0.009,0.009,0.010,0.008,0.009,0.006,0.008,0.005,0.006
market_late_rate_30d,0.052,0.004,0.047,0.074,0.083,0.064,0.051,0.042,0.048,0.059,0.058,0.076,0.133,0.066,0.118,0.202,0.091,0.091,0.024
actual_late_rate,0.029,0.030,0.046,0.066,0.030,0.030,0.028,0.029,0.044,0.042,0.124,0.075,0.057,0.141,0.190,0.045,0.066,0.012,0.034


**Reading.**
* Seller cold start is very common in early 2017 and drops to a stable share later, so the seller-history features mean little for the earliest training months. The `seller_resolved_n_min` and `seller_cold_start` features tell the model when to trust them.
* Route cold start is rare.
* `market_late_rate_30d` follows the realised late rate with a lag. This is the only legitimate way to carry drift information into a checkout-time model.

## 8. Why the as-of construction matters: a deliberately leaky seller feature
We compare three seller late rates as single-feature scorers of `late_day`, on **training-period orders only**:
1. **Leaky**: the seller's late rate over *all* orders in the dataset, which includes the order itself and its future.
2. **Naive history**: the proposal's rule, counting only orders *delivered* before purchase. It ignores late orders that are still undelivered.
3. **As-of (ours)**: `seller_late_rate_max`.

In [10]:
from olist.dataset import load_sources
src = load_sources()
pairs = src["items"][["order_id", "seller_id"]].drop_duplicates()
lab = df[["order_id", "late_day", "order_delivered_customer_date", P]]
pl = pairs.merge(lab, on="order_id", validate="m:1")
leaky_seller = pl.dropna(subset=["late_day"]).groupby("seller_id").late_day.mean()
leaky = pl.assign(r=pl.seller_id.map(leaky_seller)).groupby("order_id").r.max()

# naive: delivered-before-purchase, computed by brute-force merge_asof on delivery time
from olist.history import asof_counts
ev = pl.dropna(subset=["late_day"])[["seller_id", "order_delivered_customer_date", "late_day"]]
q = pl[["order_id", "seller_id", P]].reset_index(drop=True)
nv = asof_counts(q, ev, key="seller_id", t_col=P, event_time="order_delivered_customer_date", value="late_day")
prior = q.order_id.map(df.set_index("order_id").history_prior).to_numpy()
naive = (q.assign(r=(nv.sum_all + C.SMOOTHING_M * prior) / (nv.n_all + C.SMOOTHING_M)).groupby("order_id").r.max())

train = pop[assign_split(pop[P]) == "train"].set_index("order_id")
y = train.late_day
cmp = pd.DataFrame({name: {"ROC-AUC": roc_auc_score(y, s.reindex(y.index)), "PR-AUC": average_precision_score(y, s.reindex(y.index))}
                    for name, s in [("1 leaky (full-data rate)", leaky), ("2 naive (delivered-before)", naive),
                                    ("3 as-of (known_at)", train.seller_late_rate_max)]}).T.round(4)
print(f"training orders: {len(y):,}, base rate {y.mean():.2%} (PR-AUC of a random scorer)")
cmp

training orders: 57,050, base rate 6.63% (PR-AUC of a random scorer)


,ROC-AUC,PR-AUC
1 leaky (full-data rate),0.7086,0.1803
2 naive (delivered-before),0.5535,0.0823
3 as-of (known_at),0.5515,0.0814


In [11]:
gap = (train.seller_late_rate_max - naive.reindex(train.index))
print(f"as-of minus naive seller rate: mean {gap.mean():+.4f}; among late orders {gap[y==1].mean():+.4f}; among on-time {gap[y==0].mean():+.4f}")

as-of minus naive seller rate: mean +0.0109; among late orders +0.0113; among on-time +0.0109


**Reading (from the numbers above).**
* The **leaky** full-data rate scores far higher than either honest version. A model given it would look strong offline and fail in deployment, because it partly *contains the label*. This is the size of the error the as-of construction prevents.
* The **naive** and **as-of** rates rank orders almost identically. The as-of rate is slightly higher on average, because it also counts late orders that are not yet delivered, but the shift is the same for late and on-time orders. The Stage 1 concern was therefore real as a *level* bias but makes no practical difference to *ranking*. We keep the as-of rule because it is the correct one and costs nothing extra.
* **Seller history alone is a weak signal** (ROC-AUC ≈ 0.55 on training orders). It should not be expected to carry the model. Its value has to be shown in combination with other features and in the ablations of Stage 5.

## 9. Provisional chronological split
The split is on purchase date. A model scored on a period may train only on orders whose label would exist when that period starts. The purge must not look at the outcome, so a row is kept only if its **promised day has fully passed** (`estimated + 1 day <= cutoff`, `olist.split.matured_by`). Below we show why the outcome-based alternative (`known_at < cutoff`) is wrong.

In [12]:
pop["split"] = assign_split(pop[P])
summ = split_summary(pop)
summ["purged_if_training_ends_at_period_end"] = summ.orders - summ.matured_by_period_end
summ

,purchase_from,purchase_to_excl,orders,late,late_rate_%,matured_by_period_end,late_rate_matured_%,purged_if_training_ends_at_period_end
split,,,,,,,,
train,2017-01-01,2018-03-01,57050,3782,6.63,50356,5.64,6694
val,2018-03-01,2018-05-01,13801,1634,11.84,8442,16.81,5359
test,2018-05-01,2018-08-01,18944,722,3.81,14174,3.97,4770


In [13]:
train_rows = pop[pop.split == "train"]
cut = C.SPLITS["val"][0]
rules = {"outcome-based: known_at < cutoff (WRONG)": label_known_by(train_rows, cut),
         "maturity: estimated + 1 day <= cutoff (USED)": matured_by(train_rows, cut)}
print(f"train rows: {len(train_rows):,}, true late rate {train_rows.late_day.mean():.2%}; cutoff {cut.date()}")
display(pd.DataFrame({name: {"kept": int(k.sum()), "purged": int((~k).sum()),
                             "late_rate_kept_%": round(train_rows[k].late_day.mean() * 100, 2),
                             "late_rate_purged_%": round(train_rows[~k].late_day.mean() * 100, 2)}
                      for name, k in rules.items()}).T)
assert (train_rows.loc[rules["maturity: estimated + 1 day <= cutoff (USED)"], "known_at"] <= cut).all()
purged = train_rows[~rules["maturity: estimated + 1 day <= cutoff (USED)"]]
print("maturity-purged rows by purchase month:", purged.ym.value_counts().sort_index().loc[lambda s: s > 0].to_dict())
print(f"late rate of ALL Feb-2018 orders: {train_rows[train_rows.ym == '2018-02'].late_day.mean():.2%} "
      f"vs maturity-purged rows: {purged.late_day.mean():.2%}")
trval = pop[pop.split.isin(["train", "val"])]
keep2 = matured_by(trval, C.SPLITS["test"][0])
print(f"train+val rows usable when scoring test: {keep2.sum():,} of {len(trval):,} -> purged {(~keep2).sum():,}")
print("customers appearing in both train and test:",
      len(set(pop.loc[pop.split=="train","customer_unique_id"]) & set(pop.loc[pop.split=="test","customer_unique_id"])))

train rows: 57,050, true late rate 6.63%; cutoff 2018-03-01


,kept,purged,late_rate_kept_%,late_rate_purged_%
outcome-based: known_at < cutoff (WRONG),53511.0,3539.0,5.31,26.59
maturity: estimated + 1 day <= cutoff (USED),50356.0,6694.0,5.64,14.06


maturity-purged rows by purchase month: {Period('2017-12', 'M'): 8, Period('2018-01', 'M'): 648, Period('2018-02', 'M'): 6038}
late rate of ALL Feb-2018 orders: 14.13% vs maturity-purged rows: 14.06%


train+val rows usable when scoring test: 65,490 of 70,851 -> purged 5,361


customers appearing in both train and test: 313


**Reading.**
* Purging on `known_at` keeps recent orders only when they were delivered early. It drops a sample that is mostly late, so the training labels would be biased toward "on time".
* The maturity rule drops recent orders based only on their promised date, which is known at checkout. Its purged rows have the same late rate as the period they come from (check below). It removes a *period*, not an *outcome*. The kept rows have a lower late rate only because February 2018 was a spike month.
* The cost is losing the most recent ~3–4 weeks of training data, so the effective training window ends about one promise-length before the period being scored.
* Validation (Mar–Apr 2018) contains the March spike, while test (May–Jul 2018) has a much lower base rate. **Calibration and threshold choices made on validation may not transfer.** This is a documented risk to revisit after the EDA (Stage 4), for example by using walk-forward validation folds.

## 10. Save

In [14]:
C.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
keep_cols = (["order_id", "customer_unique_id", P, "order_estimated_delivery_date", "order_delivered_customer_date",
              "order_status", "outcome_class", "known_at", "late_day", "late_ts", "promise_failed",
              "eligible_primary", "eligible_sensitivity", "history_prior"]
             + [c for c in df.columns if c.startswith("dq_")] + FEATURES.index.tolist())
df[keep_cols].to_parquet(C.ORDER_DATASET_PATH, index=False)
print(f"saved {C.ORDER_DATASET_PATH.relative_to(ROOT)}: {len(df):,} rows x {len(keep_cols)} cols, "
      f"{C.ORDER_DATASET_PATH.stat().st_size/1e6:.1f} MB | features: {len(FEATURES)} | runtime {time.perf_counter()-T0:.0f}s | RSS {rss()}")

saved data\processed\orders_model.parquet: 98,666 rows x 74 cols, 20.5 MB | features: 57 | runtime 28s | RSS 665 MB
